# OpportunityMatch AI — Week 02
## Semantic Search for Student Opportunities

**Goal:** help students find relevant hackathons, internships, workshops, scholarships, fellowships, and projects using natural-language queries.

This notebook documents the pipeline and experiments. The implementation lives in `task.py`; this notebook imports and explores it rather than duplicating the full application.

**Important:** run `python task.py` in the VS Code terminal first and let the model download and database ingestion finish. Then run this notebook from the `week-02` folder using the same Python virtual environment.


## 1. Problem and approach

A student may search with a sentence such as *“I want to learn AI with Python”* rather than knowing the exact title of an opportunity.

The system:
1. loads and validates a CSV dataset;
2. cleans text fields;
3. splits opportunity information into chunks;
4. converts chunks into embedding vectors;
5. stores vectors and metadata in ChromaDB;
6. retrieves and ranks relevant results;
7. evaluates ranking quality with labeled example queries.

The current dataset is a small synthetic demonstration dataset, not a live directory of verified opportunities.


## 2. Imports and dataset loading

Run the notebook cells from the `week-02` directory so that `task.py` and `data/opportunities.csv` can be found.


In [13]:
import sys
!{sys.executable} -m pip install chromadb langchain-text-splitters


^C


In [1]:
from pathlib import Path
import pandas as pd
from task import load_data, TextProcessor, OpportunitySearchEngine, evaluate

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "opportunities.csv"

df = load_data(DATA_PATH)
print("Dataset path:", DATA_PATH)
print("Rows and columns:", df.shape)
print("Category count:", df["category"].nunique())
df.head(5)


ModuleNotFoundError: No module named 'sentence_transformers'

## 3. Inspect the dataset

Check the column names, missing values, duplicate IDs, and distribution across categories. These checks help catch data-quality problems before embedding.


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate IDs:", df["id"].duplicated().sum())

print("\nOpportunities per category:")
display(df["category"].value_counts().rename_axis("category").to_frame("count"))


## 4. Cleaning and text chunking

The `load_data` function cleans the text fields. `TextProcessor` combines each opportunity's title, category, description, skills, and eligibility, then splits the text into manageable chunks for retrieval.


In [ ]:
processor = TextProcessor()

sample_row = df.iloc[0]
sample_chunks = processor.process(sample_row)

print("Sample opportunity:", sample_row["title"])
print("Number of chunks:", len(sample_chunks))
for number, chunk in enumerate(sample_chunks, start=1):
    print(f"\nChunk {number}:\n{chunk}")


## 5. Embeddings and vector database

The embedding model represents text as numeric vectors so that semantically similar text can be retrieved even when the query and description do not use exactly the same words.

ChromaDB stores the vectors alongside metadata such as title, category, skills, and eligibility.

**Run this section only after `python task.py` has completed its first model download and created the database.** If you run the notebook before that, model loading may start another download.


In [ ]:
# This loads the embedding model and connects to the persistent ChromaDB database.
# It does not re-ingest the CSV.
engine = OpportunitySearchEngine()

# Confirm that the collection created by task.py is available.
engine.collection = engine.client.get_collection("student_opportunities")
print("Stored text chunks:", engine.collection.count())


## 6. Search experiments

Try different types of natural-language queries. Inspect whether the returned opportunities are actually useful—not just whether the program returns three items.


In [ ]:
test_queries = [
    "I want to learn AI with Python",
    "Find cybersecurity opportunities for students",
    "I need funding for student research",
    "I want to build websites using React",
    "I want to contribute to open-source software",
]

for query in test_queries:
    print("\n" + "=" * 70)
    print("QUERY:", query)
    results = engine.search(query, top_k=3)
    engine.display_results(results)


## 7. Category-filter experiment

A category filter restricts results to one opportunity category. The category must match a value in the CSV.


In [ ]:
print("Available categories:")
print(sorted(df["category"].unique()))

# Change "Workshop" to another category from the list above if desired.
category_results = engine.search(
    "I want to learn technology skills",
    top_k=3,
    category="Workshop"
)
engine.display_results(category_results)


## 8. Evaluation

The evaluation function reports Recall@3 and Mean Reciprocal Rank (MRR) against the labeled examples defined in `task.py`.

- **Recall@3:** the fraction of labeled relevant opportunities retrieved in the first three results.
- **Reciprocal Rank:** `1 / rank` for the first relevant result, or `0` if none is found.
- **MRR:** the average reciprocal rank across the test queries.

The current labels are starter labels. Manually inspect results and correct the relevant-ID sets before using the metrics in a report. Small datasets produce noisy estimates.


In [ ]:
evaluate(engine)


## 9. Limitations and next improvements

### Current limitations
- The dataset contains only 20 sample opportunities and is not live or externally verified.
- Relevance labels are manually assigned and should be reviewed.
- The lexical score is simple token overlap, not a learned ranking model.
- A high similarity score does not guarantee eligibility, availability, or legitimacy.

### Planned improvements
1. Add more realistic, verified records and track source URLs and update dates.
2. Compare semantic-only ranking with hybrid ranking using the same labeled queries.
3. Add a cross-encoder reranker and evaluate whether it improves ranking.
4. Add automated tests for empty queries, missing columns, unknown categories, and duplicate IDs.
5. Consider a small API or web interface after the retrieval pipeline is reliable.

## 10. Conclusion

OpportunityMatch AI demonstrates an end-to-end retrieval workflow: data validation, text preparation, embeddings, vector storage, semantic search, hybrid ranking, and evaluation. The next step is to measure ranking quality carefully and improve it using verified data and repeatable tests.
